# Part 2: Results Analysis & Discussion
Notebook này trình bày kết quả phân tích và so sánh mô hình trên bộ dữ liệu thực.

## 1. Setup

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Import các hàm tự định nghĩa từ source code của các thành viên
from data_pipeline import DataPipeline, load_data, train_test_split
from model_comparison import run_diagnostics, train_models, hyperparameter_tuning, comparison_table, plot_coefficients
from advanced_methods import kernel_ridge_fit

plt.style.use('seaborn-v0_8-whitegrid')
%matplotlib inline

## 2. Khám phá dữ liệu (EDA)

In [ ]:
# 1. Load dữ liệu từ file CSV thô
df_raw = pd.read_csv("data/melbourne_housing.csv")

# 2. Thực hiện train_test_split ngay từ Dataframe thô (Tỷ lệ 70:30)
# Giữ chặt df_test_raw, không cho bất kỳ hàm tiền xử lý nào chạm vào ở Phase 1
np.random.seed(42)
msk = np.random.rand(len(df_raw)) < 0.7
df_train_raw = df_raw[msk].copy()
df_test_raw = df_raw[~msk].copy()

print(f"Kích thước tập Train thô: {df_train_raw.shape}")
print(f"Kích thước tập Test thô: {df_test_raw.shape}")

## Phase 1: Vòng lặp xử lí dữ liệu gốc

### 1. Tiền xử lý dữ liệu

In [ ]:
# TODO: Preprocessing pipeline
# Khởi tạo và chạy pipeline trên tập train thô
pipeline = DataPipeline(drop_columns=[])
X_train_raw, y_train = pipeline.fit_transform(df_train_raw)
feature_names_raw = pipeline.encoded_columns

print(f"Số lượng đặc trưng ban đầu sau One-Hot: {len(feature_names_raw)}")

## 2. Chẩn đoán sơ khởi

In [ ]:
# 5. Gọi hàm của Member B để lấy đầy đủ các chỉ số Toán học từ Phần 1
from model_comparison import (
    custom_ols_fit,
    custom_vif,
    custom_inference,
)  # Các lõi Toán Phần 1

diagnostics_df = run_diagnostics(
    X_train_raw=X_train_raw,
    y_train=y_train,
    feature_names=feature_names_raw,
    custom_ols_func=custom_ols_fit,
    custom_vif_func=custom_vif,
    custom_inference_func=custom_inference,
)

# Hiển thị bảng kết quả cho Data Analyst xem xét
diagnostics_df.sort_values(by="VIF", ascending=False).head(10)

### Sync point: Đánh giá & quyết định
* **Nhận xét của Data Analyst:** Dựa vào bảng chẩn đoán trên, các biến `Rooms` và `Bedroom2` có chỉ số $VIF > 10$, đồng thời hệ số p-value của `Bedroom2` lớn hơn 0.05 (không có ý nghĩa thống kê).
* **Quyết định trảm biến:** Nhóm thống nhất loại bỏ cột `Bedroom2` ra khỏi hệ thống để làm sạch ma trận đặc trưng.

## Phase 2: Huấn luyện & Đánh giá mô hình

### 1. Tiền xử lí dữ liệu cho mô hình

In [ ]:
# Khởi tạo Pipeline mới nhận lệnh loại bỏ cột từ Data Analyst
drop_list = ["Bedroom2"] # Example
pipeline_best = DataPipeline(drop_columns=drop_list)

# Chạy fit_transform trên Train thô và transform trên Test thô
X_train_best, y_train = pipeline_best.fit_transform(df_train_raw)
X_test_best, y_test = pipeline_best.transform(df_test_raw)

# Đồng thời tạo ra ma trận Test hệ đầy đủ biến phục vụ cho mô hình OLS Baseline
X_test_raw, _ = pipeline.transform(df_test_raw)

# Đóng gói từ điển Metadata bàn giao kết quả (Contract 1)
metadata = {
    "feature_names": pipeline_best.encoded_columns,
    "target_name": "Price",
    "pipeline": pipeline_best,
    "imputation_strategy": "knn_imputer",
    "scaling_method": "standardize",
}

### 2. Tìm Siêu Tham Số Ridge Qua K-Fold CV

In [ ]:
# ML Modeler tiến hành dò tìm thông số phạt tốt nhất trên tập X_train_best
lambda_grid = [0.01, 0.1, 1.0, 5.0, 10.0, 50.0, 100.0]
from model_comparison import custom_ridge_fit  # Hàm lõi phần 1

best_lambda_ridge, cv_scores = hyperparameter_tuning(
    X_train=X_train_best,
    y_train=y_train,
    model_class=custom_ridge_fit,
    param_grid={"alpha": lambda_grid},
    k=5,
)
print(f"Hệ số phạt tối ưu cho Ridge tìm được: \u03bb = {best_lambda_ridge}")

### 3. Huấn luyện mô hình

In [ ]:
# 6. Chạy 4 mô hình song song hoàn toàn bằng code tự chế (Mở két sắt chứa tập Test để chấm điểm)
results_dict = train_models(
    X_train_raw=X_train_raw,
    X_train_best=X_train_best,
    y_train=y_train,
    X_test_raw=X_test_raw,
    X_test_best=X_test_best,
    y_test=y_test,
    custom_ols_func=custom_ols_fit,
    custom_ridge_func=custom_ridge_fit,
    custom_kernel_func=kernel_ridge_fit,  # Nhận hàm core Toán phi tuyến từ Member C
    lambda_ridge=best_lambda_ridge,
    lambda_kernel=1.0,  # Giả định lambda cho mô hình Kernel nâng cao
)

### 4. DataFrame

In [ ]:
# ML Modeler chuyển giao bảng xếp hạng hiệu năng tổng hợp cho Data Analyst
df_leaderboard = comparison_table(results_dict)
df_leaderboard

## Phase 3: Đánh giá phần dư và trực quan hóa
* **Mục tiêu:** Vẽ biểu đồ chẩn đoán sức khỏe phần dư của mô hình vô địch, trực quan hóa mức độ quan trọng của đặc trưng nhằm viết báo cáo Typst thực tế.

In [ ]:
# 1. Vẽ biểu đồ hệ số thanh ngang giải thích tầm quan trọng của các biến sạch
plot_coefficients(results=results_dict, feature_names=metadata["feature_names"])

# 2. Trích xuất phần dư của mô hình tốt nhất để phân tích kiểm định Gauss-Markov
from model_comparison import evaluate_gauss_markov_assumptions, plot_predictions

best_model_key = "Ridge_custom"
best_residuals = y_test - results_dict[best_model_key]["predictions_test"]

gm_metrics = evaluate_gauss_markov_assumptions(X_train_best, y_train, best_residuals)
plot_predictions(y_test, results_dict)

## Phase 4: Biện luận & Tổng kết báo cáo (Cell này nên bỏ, tập trung viết bên Typst)
*(Data Analyst sử dụng các kết quả định lượng trên cell để viết báo cáo khoa học)*
1. **Lý do Ridge vượt trội OLS:** Giải thích dựa trên việc bóp nghẹt phương sai của các biến tương quan cao.
2. **Ý nghĩa thực tế của hệ số hồi quy:** Phân tích các biến động vị trí ảnh hưởng trực tiếp thế nào tới giá nhà đất tại Melbourne.